# Limpieza de datos capa Silver

# Montar Drive

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Instalar/actualizar librerías necesarias

In [2]:
!pip -q install geopandas pyarrow shapely fiona

# Imports y rutas base

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
import geopandas as gpd
import re
import unicodedata
import json

BASE_DIR = Path("/content/drive/MyDrive/AI Projects/DeepWave Canarias")
BRONZE_DIR = BASE_DIR / "data"
SILVER_DIR = BASE_DIR / "silver"

OUT_DIR = SILVER_DIR / "beach_geography"
QC_DIR = SILVER_DIR / "_quality_reports"

OUT_DIR.mkdir(parents=True, exist_ok=True)
QC_DIR.mkdir(parents=True, exist_ok=True)

BBOX_CANARIAS = {
    "lat_min": 27.0,
    "lat_max": 29.5,
    "lon_min": -18.5,
    "lon_max": -13.0,
}

print("BASE_DIR:", BASE_DIR)
print("Existe BASE_DIR:", BASE_DIR.exists())
print("Existe BRONZE_DIR:", BRONZE_DIR.exists())

if not BRONZE_DIR.exists():
    raise FileNotFoundError(
        "No existe la carpeta data/. Revisa que Google Drive esté montado y que BASE_DIR sea correcta."
    )

BASE_DIR: /content/drive/MyDrive/AI Projects/DeepWave Canarias
Existe BASE_DIR: True
Existe BRONZE_DIR: True


# Utilidades generales

In [4]:
def normalize_text(value):
    if pd.isna(value):
        return np.nan

    value = str(value).strip()
    value = unicodedata.normalize("NFKD", value)
    value = "".join(c for c in value if not unicodedata.combining(c))
    value = re.sub(r"\s+", " ", value)

    return value.upper()


def slugify(value):
    value = normalize_text(value)

    if pd.isna(value):
        return "UNKNOWN"

    value = re.sub(r"[^A-Z0-9]+", "_", value)
    value = re.sub(r"_+", "_", value)

    return value.strip("_")


def infer_column(df, candidates):
    """
    Busca una columna por nombre aproximado.
    Tolera acentos, mayúsculas, minúsculas y espacios raros.
    """

    cols_norm = {
        normalize_text(col): col
        for col in df.columns
    }

    for candidate in candidates:
        candidate_norm = normalize_text(candidate)

        for col_norm, original_col in cols_norm.items():
            if candidate_norm == col_norm:
                return original_col

        for col_norm, original_col in cols_norm.items():
            if candidate_norm in col_norm:
                return original_col

    return None


def parse_coordinate(value):
    """
    Convierte coordenadas en formato decimal o grados/minutos/segundos a float.
    Soporta coma decimal y hemisferios N/S/E/W/O.
    """

    if pd.isna(value):
        return np.nan

    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value)

    s = str(value).strip().upper()
    s = s.replace(",", ".")

    if s in ["", "NAN", "NONE", "NULL"]:
        return np.nan

    sign = 1

    if any(h in s for h in ["W", "O", "S"]):
        sign = -1

    if s.startswith("-"):
        sign = -1

    nums = re.findall(r"-?\d+(?:\.\d+)?", s)

    if not nums:
        return np.nan

    try:
        if len(nums) >= 3:
            deg = abs(float(nums[0]))
            minutes = float(nums[1])
            seconds = float(nums[2])
            value = deg + minutes / 60 + seconds / 3600

        elif len(nums) == 2 and ("º" in s or "°" in s or "'" in s):
            deg = abs(float(nums[0]))
            minutes = float(nums[1])
            value = deg + minutes / 60

        else:
            value = abs(float(nums[0])) if sign == -1 else float(nums[0])

        return sign * abs(value) if sign == -1 else value

    except Exception:
        return np.nan

# Buscar automáticamente archivos de entrada

In [5]:
# CSV de playas
candidatos_playas = list(BRONZE_DIR.rglob("*Playas*.csv"))

print("Candidatos CSV de playas encontrados:")
for p in candidatos_playas:
    print("-", p)

if not candidatos_playas:
    raise FileNotFoundError(
        "No se encontró ningún CSV de playas dentro de data/."
    )

PATH_PLAYAS = candidatos_playas[0]

# GeoJSON GRAFCAN
candidatos_islas = list(BRONZE_DIR.rglob("islas.geojson"))
candidatos_municipios = list(BRONZE_DIR.rglob("municipios.geojson"))

if not candidatos_islas:
    raise FileNotFoundError("No se encontró islas.geojson dentro de data/.")

if not candidatos_municipios:
    raise FileNotFoundError("No se encontró municipios.geojson dentro de data/.")

PATH_ISLAS = candidatos_islas[0]
PATH_MUNICIPIOS = candidatos_municipios[0]

# MITECO, por ahora referencia documental
candidatos_miteco = list(BRONZE_DIR.rglob("*miteco*")) + list(BRONZE_DIR.rglob("*MITECO*"))
PATH_MITECO = candidatos_miteco[0] if candidatos_miteco else None

print("\nUsando PATH_PLAYAS:")
print(PATH_PLAYAS)

print("\nUsando PATH_ISLAS:")
print(PATH_ISLAS)

print("\nUsando PATH_MUNICIPIOS:")
print(PATH_MUNICIPIOS)

print("\nPATH_MITECO:")
print(PATH_MITECO)

Candidatos CSV de playas encontrados:
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/data/bronze/Guía de Playas de España/Playas_espa%C3%B1olas.csv

Usando PATH_PLAYAS:
/content/drive/MyDrive/AI Projects/DeepWave Canarias/data/bronze/Guía de Playas de España/Playas_espa%C3%B1olas.csv

Usando PATH_ISLAS:
/content/drive/MyDrive/AI Projects/DeepWave Canarias/data/bronze/GRAFCAN IDECanarias/islas.geojson

Usando PATH_MUNICIPIOS:
/content/drive/MyDrive/AI Projects/DeepWave Canarias/data/bronze/GRAFCAN IDECanarias/municipios.geojson

PATH_MITECO:
/content/drive/MyDrive/AI Projects/DeepWave Canarias/data/bronze/MITECO/fuente_miteco_inundacion_costera.txt.rtf


# Leer CSV de playas

In [6]:
encodings = ["utf-8", "utf-8-sig", "latin1", "cp1252"]
separators = [";", ",", "\t"]

playas = None
read_info = None

for enc in encodings:
    for sep in separators:
        try:
            tmp = pd.read_csv(PATH_PLAYAS, encoding=enc, sep=sep)

            if tmp.shape[1] > 3:
                playas = tmp
                read_info = {
                    "encoding": enc,
                    "sep": sep,
                    "rows": tmp.shape[0],
                    "columns": tmp.shape[1],
                }
                break

        except Exception:
            continue

    if playas is not None:
        break

if playas is None:
    raise ValueError(
        "No se pudo leer el CSV de playas con las codificaciones y separadores probados."
    )

print("Lectura correcta:")
print(read_info)

print("\nShape:")
print(playas.shape)

print("\nColumnas originales:")
print(playas.columns.tolist())

display(playas.head())

Lectura correcta:
{'encoding': 'utf-8', 'sep': ',', 'rows': 3554, 'columns': 80}

Shape:
(3554, 80)

Columnas originales:
['X', 'Y', 'OBJECTID', 'Comunidad_', 'Provincia', 'Isla', 'Código_IN', 'Término_M', 'Web_munici', 'Identifica', 'Nombre', 'Nombre_alt', 'Nombre_a_1', 'Descripci', 'Longitud', 'Anchura', 'Variación', 'Grado_ocup', 'Grado_urba', 'Paseo_mar', 'Tipo_paseo', 'Tipo_de_ar', 'Condicione', 'Zona_fonde', 'Nudismo', 'Vegetació', 'Vegetaci_1', 'Actuacione', 'Actuacio_1', 'Bandera_az', 'Auxilio_y_', 'Auxilio_y1', 'Señalizac', 'Señaliza_', 'Forma_de_a', 'Señaliza1', 'Acceso_dis', 'Carretera_', 'Autobús', 'Autobús_t', 'Aparcamien', 'Aparcami_1', 'Aparcami_2', 'Aseos', 'Lavapies', 'Duchas', 'Teléfonos', 'Papelera', 'Servicio_l', 'Alquiler_s', 'Alquiler_h', 'Alquiler_n', 'Oficina_tu', 'Establecim', 'Establec_1', 'Zona_infan', 'Zona_depor', 'Club_naút', 'Submarinis', 'Zona_Surf', 'Observacio', 'Coordenada', 'Coordena_1', 'Huso', 'Coordena_2', 'Coordena_3', 'Puerto_dep', 'Web_puerto',

,X,Y,OBJECTID,Comunidad_,Provincia,Isla,Código_IN,Término_M,Web_munici,Identifica,...,Dirección,Teléfono_,Distancia1,Composici,Fachada_Li,Espacio_pr,Espacio__1,Coordena_4,Coordena_5,URL_MAGRAM
0,-543984.9557,4.370555e+06,1,Andalucía,Málaga,,29069,Marbella,http://www.marbella.es,316.0,...,"A-7 Km. 186,7 (Marbella)",951976669,6 km.,Arena,Urbana,No,,-4.8867,36.5066,https://sig.miteco.gob.es/93/ClienteWS/Guia-Pl...
1,-529891.9081,4.367771e+06,2,Andalucía,Málaga,,29069,Marbella,http://www.marbella.es,318.0,...,"A-7 Km. 186,7 (Marbella)",951976669,7 km.,Arena / Grava,Urbana,No,,-4.7601,36.4865,https://sig.miteco.gob.es/93/ClienteWS/Guia-Pl...
2,-524448.3850,4.367937e+06,3,Andalucía,Málaga,,29070,Mijas,http://www.mijas.es,330.0,...,"A-7 Km. 186,7 (Marbella)",951976669,10 km.,Arena / Roca / Grava,Urbana,Sí,Zona de Especial Conservación Calahonda (ES617...,-4.7112,36.4877,https://sig.miteco.gob.es/93/ClienteWS/Guia-Pl...
3,-517145.8264,4.370638e+06,4,Andalucía,Málaga,,29070,Mijas,http://www.mijas.es,402904.0,...,"A-7 Km. 186,7 (Marbella)",951976669,18 km.,Arena / Roca,Semiurbana,Sí,Zona de Especial Conservación Calahonda (ES617...,-4.6456,36.5072,https://sig.miteco.gob.es/93/ClienteWS/Guia-Pl...
4,-4975.9812,4.664878e+06,5,Comunitat Valenciana,Alicante/Alacant,,3018,Altea,http://www.altea.es,450301.0,...,"Doctor Ramón y Cajal, 7 (Benidorm)",966878787,11 km.,Bolos / Grava,Urbana,No,,-0.0447,38.6024,https://sig.miteco.gob.es/93/ClienteWS/Guia-Pl...


# Detectar columnas del CSV

In [7]:
def first_existing_column(df, candidates):
    for col in candidates:
        if col in df.columns:
            return col
    return None


col_nombre = first_existing_column(playas, ["Nombre"]) or infer_column(
    playas,
    [
        "nombre",
        "nombre playa",
        "playa",
        "denominacion",
        "denominacion playa",
        "nombre de la playa",
    ],
)

col_isla = first_existing_column(playas, ["Isla"]) or infer_column(
    playas,
    [
        "isla",
        "nombre isla",
        "isla nombre",
    ],
)

col_municipio = first_existing_column(
    playas,
    [
        "Término_M",
        "Termino_M",
        "Término M",
        "Termino M",
    ],
) or infer_column(
    playas,
    [
        "municipio",
        "nombre municipio",
        "ayuntamiento",
        "termino municipal",
        "término municipal",
        "termino_m",
        "término_m",
    ],
)

col_provincia = first_existing_column(playas, ["Provincia"]) or infer_column(
    playas,
    [
        "provincia",
        "nombre provincia",
    ],
)

col_ccaa = first_existing_column(playas, ["Comunidad_"]) or infer_column(
    playas,
    [
        "comunidad",
        "comunidad autonoma",
        "comunidad autónoma",
        "ccaa",
        "autonomia",
        "autonomía",
    ],
)

# En este CSV, Coordena_4 = longitud decimal y Coordena_5 = latitud decimal.
# No usar X/Y porque son coordenadas proyectadas.
# No usar Longitud porque es la longitud física de la playa.
col_lon = first_existing_column(
    playas,
    [
        "Coordena_4",
        "longitud_decimal",
        "lon",
        "lng",
        "longitude",
    ],
)

col_lat = first_existing_column(
    playas,
    [
        "Coordena_5",
        "latitud_decimal",
        "lat",
        "latitude",
    ],
)

detected_cols = {
    "nombre": col_nombre,
    "isla": col_isla,
    "municipio": col_municipio,
    "provincia": col_provincia,
    "ccaa": col_ccaa,
    "lat": col_lat,
    "lon": col_lon,
}

print("Diccionario de columnas detectadas corregido:")
detected_cols

Diccionario de columnas detectadas corregido:


{'nombre': 'Nombre',
 'isla': 'Isla',
 'municipio': 'Término_M',
 'provincia': 'Provincia',
 'ccaa': 'Comunidad_',
 'lat': 'Coordena_5',
 'lon': 'Coordena_4'}

# Validar columnas mínimas

In [8]:
if col_nombre is None:
    print("AVISO: no se detectó columna de nombre de playa. Se usará 'ZONA_SIN_NOMBRE'.")

if col_municipio is None:
    print("AVISO: no se detectó columna de municipio. Se usará 'MUNICIPIO_DESCONOCIDO'.")

if col_lat is None or col_lon is None:
    raise ValueError(
        "No se han detectado columnas válidas de latitud/longitud. "
        "En este CSV deberían existir Coordena_5 y Coordena_4."
    )

test_lat = playas[col_lat].apply(parse_coordinate)
test_lon = playas[col_lon].apply(parse_coordinate)

print("Columna latitud:", col_lat)
print("Columna longitud:", col_lon)

print("\nRango latitud detectado:")
print(test_lat.describe())

print("\nRango longitud detectado:")
print(test_lon.describe())

mask_test_bbox = (
    test_lat.between(BBOX_CANARIAS["lat_min"], BBOX_CANARIAS["lat_max"])
    & test_lon.between(BBOX_CANARIAS["lon_min"], BBOX_CANARIAS["lon_max"])
)

print("\nFilas dentro del bbox de Canarias:", int(mask_test_bbox.sum()))

if mask_test_bbox.sum() == 0:
    raise ValueError(
        "Las coordenadas detectadas no caen dentro del bbox de Canarias. "
        "Revisar columnas de lat/lon."
    )

print("\nColumnas mínimas correctas.")

Columna latitud: Coordena_5
Columna longitud: Coordena_4

Rango latitud detectado:
count    3551.000000
mean       38.698540
std         5.008371
min        27.640800
25%        36.763500
50%        39.938400
75%        42.543000
max        43.771500
Name: Coordena_5, dtype: float64

Rango longitud detectado:
count    3551.000000
mean       -5.108315
std         6.104853
min       -18.151200
25%        -8.851850
50%        -4.825800
75%         0.068850
max         4.303000
Name: Coordena_4, dtype: float64

Filas dentro del bbox de Canarias: 561

Columnas mínimas correctas.


# Normalizar coordenadas y filtrar Canarias

In [9]:
df = playas.copy()

df["_lat"] = df[col_lat].apply(parse_coordinate)
df["_lon"] = df[col_lon].apply(parse_coordinate)

mask_bbox = (
    df["_lat"].between(BBOX_CANARIAS["lat_min"], BBOX_CANARIAS["lat_max"])
    & df["_lon"].between(BBOX_CANARIAS["lon_min"], BBOX_CANARIAS["lon_max"])
)

mask_text = pd.Series(False, index=df.index)

for col in [col_ccaa, col_provincia, col_isla]:
    if col is not None:
        col_norm = df[col].astype(str).map(normalize_text)
        mask_text = mask_text | col_norm.str.contains(
            "CANARIAS|PALMAS|SANTA CRUZ|TENERIFE|GRAN CANARIA|LANZAROTE|FUERTEVENTURA|GOMERA|HIERRO|PALMA|GRACIOSA|ALEGRANZA",
            na=False,
            regex=True,
        )

# Requerimos bbox correcto. El texto se usa como validación adicional.
if mask_text.any():
    df_can = df[mask_bbox & mask_text].copy()
else:
    df_can = df[mask_bbox].copy()

print("Playas totales:", len(df))
print("Playas Canarias detectadas:", len(df_can))

print("\nRango final lat/lon:")
print(df_can[["_lat", "_lon"]].describe())

display(df_can.head())

Playas totales: 3554
Playas Canarias detectadas: 561

Rango final lat/lon:
             _lat        _lon
count  561.000000  561.000000
mean    28.403472  -15.541325
std      0.429124    1.438172
min     27.640800  -18.151200
25%     28.078000  -16.638600
50%     28.358500  -15.696700
75%     28.721900  -13.860100
max     29.385800  -13.421300


,X,Y,OBJECTID,Comunidad_,Provincia,Isla,Código_IN,Término_M,Web_munici,Identifica,...,Distancia1,Composici,Fachada_Li,Espacio_pr,Espacio__1,Coordena_4,Coordena_5,URL_MAGRAM,_lat,_lon
25,-1.823035e+06,3.285876e+06,26,Canarias,Santa Cruz de Tenerife,Tenerife,38020,Güímar,http://www.guimar.es,692.0,...,,Arena / Grava,Urbana,No,,-16.3766,28.2923,https://sig.miteco.gob.es/93/ClienteWS/Guia-Pl...,28.2923,-16.3766
488,-2.001513e+06,3.203762e+06,489,Canarias,Santa Cruz de Tenerife,El Hierro,38901,El Pinar de El Hierro,http://www.elpinardeelhierro.com,597.0,...,40 km.,Arena / Grava,Urbana,Sí,Reserva de la Biosfera,-17.9799,27.6408,https://sig.miteco.gob.es/93/ClienteWS/Guia-Pl...,27.6408,-17.9799
489,-2.017310e+06,3.219592e+06,490,Canarias,Santa Cruz de Tenerife,El Hierro,38013,Frontera,http://www.aytofrontera.org,601.0,...,58 km.,Arena / Roca,Acantilado,Sí,LIC/ZEC/ZEPA/Hábitats Naturales de Interés Com...,-18.1218,27.7667,https://sig.miteco.gob.es/93/ClienteWS/Guia-Pl...,27.7667,-18.1218
490,-2.020582e+06,3.217126e+06,491,Canarias,Santa Cruz de Tenerife,El Hierro,38013,Frontera,http://www.aytofrontera.org,602.0,...,60 km.,Grava,Montaña,Sí,LIC/ZEC/ZEPA/Parque Rural/Reserva de la Biosfera,-18.1512,27.7471,https://sig.miteco.gob.es/93/ClienteWS/Guia-Pl...,27.7471,-18.1512
491,-2.014771e+06,3.218283e+06,492,Canarias,Santa Cruz de Tenerife,El Hierro,38013,Frontera,http://www.aytofrontera.org,603.0,...,50 km.,Roca,Acantilado,No,,-18.0990,27.7563,https://sig.miteco.gob.es/93/ClienteWS/Guia-Pl...,27.7563,-18.0990


# Crear tabla base dim_zone

In [10]:
dim_zone = pd.DataFrame()

if col_nombre is not None:
    dim_zone["nombre_zona"] = df_can[col_nombre]
else:
    dim_zone["nombre_zona"] = "ZONA_SIN_NOMBRE"

if col_municipio is not None:
    dim_zone["municipio"] = df_can[col_municipio]
else:
    dim_zone["municipio"] = "MUNICIPIO_DESCONOCIDO"

if col_isla is not None:
    dim_zone["isla"] = df_can[col_isla]
else:
    dim_zone["isla"] = "ISLA_DESCONOCIDA"

dim_zone["lat"] = df_can["_lat"]
dim_zone["lon"] = df_can["_lon"]

dim_zone["nombre_zona"] = dim_zone["nombre_zona"].fillna("ZONA_SIN_NOMBRE")
dim_zone["municipio"] = dim_zone["municipio"].fillna("MUNICIPIO_DESCONOCIDO")
dim_zone["isla"] = dim_zone["isla"].fillna("ISLA_DESCONOCIDA")

dim_zone["nombre_zona_norm"] = dim_zone["nombre_zona"].map(normalize_text)
dim_zone["municipio_norm"] = dim_zone["municipio"].map(normalize_text)
dim_zone["isla_norm"] = dim_zone["isla"].map(normalize_text)

dim_zone["tipo_zona"] = "playa"

dim_zone = dim_zone.dropna(subset=["lat", "lon"]).copy()

dim_zone = dim_zone.drop_duplicates(
    subset=["nombre_zona_norm", "municipio_norm", "lat", "lon"]
).reset_index(drop=True)

print("Shape dim_zone inicial:", dim_zone.shape)

print("\nMunicipios desconocidos:")
print((dim_zone["municipio"] == "MUNICIPIO_DESCONOCIDO").sum())

display(dim_zone.head())

Shape dim_zone inicial: (561, 9)

Municipios desconocidos:
0


,nombre_zona,municipio,isla,lat,lon,nombre_zona_norm,municipio_norm,isla_norm,tipo_zona
0,El Puertito,Güímar,Tenerife,28.2923,-16.3766,EL PUERTITO,GUIMAR,TENERIFE,playa
1,La Restinga,El Pinar de El Hierro,El Hierro,27.6408,-17.9799,LA RESTINGA,EL PINAR DE EL HIERRO,EL HIERRO,playa
2,Arenas Blancas,Frontera,El Hierro,27.7667,-18.1218,ARENAS BLANCAS,FRONTERA,EL HIERRO,playa
3,El Verodal,Frontera,El Hierro,27.7471,-18.1512,EL VERODAL,FRONTERA,EL HIERRO,playa
4,Charco Azul,Frontera,El Hierro,27.7563,-18.0990,CHARCO AZUL,FRONTERA,EL HIERRO,playa


# Cargar GRAFCAN

In [11]:
gdf_zonas = gpd.GeoDataFrame(
    dim_zone.copy(),
    geometry=gpd.points_from_xy(dim_zone["lon"], dim_zone["lat"]),
    crs="EPSG:4326",
)

gdf_islas = gpd.read_file(PATH_ISLAS).to_crs("EPSG:4326")
gdf_municipios = gpd.read_file(PATH_MUNICIPIOS).to_crs("EPSG:4326")

print("gdf_zonas:", gdf_zonas.shape)
print("gdf_islas:", gdf_islas.shape)
print("gdf_municipios:", gdf_municipios.shape)

print("\nColumnas islas:")
print(gdf_islas.columns.tolist())

print("\nColumnas municipios:")
print(gdf_municipios.columns.tolist())

display(gdf_islas.head())
display(gdf_municipios.head())

gdf_zonas: (561, 10)
gdf_islas: (2796, 2)
gdf_municipios: (88, 4)

Columnas islas:
['nombre', 'geometry']

Columnas municipios:
['codigo', 'nombre', 'isla', 'geometry']


,nombre,geometry
0,FUERTEVENTURA,"POLYGON ((-14.51203 28.0687, -14.51201 28.0687..."
1,FUERTEVENTURA,"POLYGON ((-14.51142 28.06935, -14.51142 28.069..."
2,FUERTEVENTURA,"POLYGON ((-14.21731 28.22443, -14.21729 28.224..."
3,FUERTEVENTURA,"POLYGON ((-14.08957 28.49836, -14.08957 28.498..."
4,FUERTEVENTURA,"POLYGON ((-14.04422 28.57846, -14.04423 28.578..."


,codigo,nombre,isla,geometry
0,38014,FUENCALIENTE,LA PALMA,"MULTIPOLYGON (((-17.84067 28.45273, -17.84067 ..."
1,38047,TIJARAFE,LA PALMA,"MULTIPOLYGON (((-17.95524 28.65686, -17.9552 2..."
2,38008,BREÑA ALTA,LA PALMA,"POLYGON ((-17.83335 28.68844, -17.83345 28.688..."
3,38030,PUNTALLANA,LA PALMA,"MULTIPOLYGON (((-17.73194 28.72529, -17.73193 ..."
4,38037,SANTA CRUZ DE LA PALMA,LA PALMA,"POLYGON ((-17.816 28.73927, -17.81607 28.73928..."


# Enriquecer isla y municipio con GRAFCAN

In [12]:
col_isla_grafcan = infer_column(
    gdf_islas.drop(columns="geometry", errors="ignore"),
    [
        "isla",
        "nombre",
        "nombre isla",
        "isla nombre",
        "denominacion",
    ],
)

col_municipio_grafcan = infer_column(
    gdf_municipios.drop(columns="geometry", errors="ignore"),
    [
        "municipio",
        "nombre",
        "nombre municipio",
        "denominacion",
    ],
)

print("Columna isla GRAFCAN:", col_isla_grafcan)
print("Columna municipio GRAFCAN:", col_municipio_grafcan)

gdf_zonas_m = gdf_zonas.to_crs("EPSG:3857")
gdf_islas_m = gdf_islas.to_crs("EPSG:3857")
gdf_municipios_m = gdf_municipios.to_crs("EPSG:3857")

# Isla más cercana
try:
    joined_islas = gpd.sjoin_nearest(
        gdf_zonas_m,
        gdf_islas_m,
        how="left",
        max_distance=10000,
        distance_col="distance_to_isla_m",
    )

    joined_islas = (
        joined_islas
        .sort_values("distance_to_isla_m")
        .groupby(level=0)
        .first()
    )

    gdf_zonas["distance_to_isla_m"] = joined_islas["distance_to_isla_m"].reindex(gdf_zonas.index)
    gdf_zonas["spatial_match_isla"] = joined_islas["index_right"].notna().reindex(gdf_zonas.index).fillna(False)

    if col_isla_grafcan is not None and col_isla_grafcan in joined_islas.columns:
        gdf_zonas["isla_spatial"] = joined_islas[col_isla_grafcan].reindex(gdf_zonas.index)
    else:
        gdf_zonas["isla_spatial"] = np.nan

except Exception as e:
    print("No se pudo hacer sjoin_nearest con islas:", e)
    gdf_zonas["distance_to_isla_m"] = np.nan
    gdf_zonas["spatial_match_isla"] = False
    gdf_zonas["isla_spatial"] = np.nan

# Municipio más cercano
try:
    joined_municipios = gpd.sjoin_nearest(
        gdf_zonas_m,
        gdf_municipios_m,
        how="left",
        max_distance=10000,
        distance_col="distance_to_municipio_m",
    )

    joined_municipios = (
        joined_municipios
        .sort_values("distance_to_municipio_m")
        .groupby(level=0)
        .first()
    )

    gdf_zonas["distance_to_municipio_m"] = joined_municipios["distance_to_municipio_m"].reindex(gdf_zonas.index)
    gdf_zonas["spatial_match_municipio"] = joined_municipios["index_right"].notna().reindex(gdf_zonas.index).fillna(False)

    if col_municipio_grafcan is not None and col_municipio_grafcan in joined_municipios.columns:
        gdf_zonas["municipio_spatial"] = joined_municipios[col_municipio_grafcan].reindex(gdf_zonas.index)
    else:
        gdf_zonas["municipio_spatial"] = np.nan

except Exception as e:
    print("No se pudo hacer sjoin_nearest con municipios:", e)
    gdf_zonas["distance_to_municipio_m"] = np.nan
    gdf_zonas["spatial_match_municipio"] = False
    gdf_zonas["municipio_spatial"] = np.nan

gdf_zonas["isla"] = gdf_zonas["isla"].fillna("ISLA_DESCONOCIDA")
gdf_zonas["municipio"] = gdf_zonas["municipio"].fillna("MUNICIPIO_DESCONOCIDO")

gdf_zonas["isla_norm"] = gdf_zonas["isla"].map(normalize_text)
gdf_zonas["municipio_norm"] = gdf_zonas["municipio"].map(normalize_text)

print("Match isla %:", round(gdf_zonas["spatial_match_isla"].mean() * 100, 2))
print("Match municipio %:", round(gdf_zonas["spatial_match_municipio"].mean() * 100, 2))

display(gdf_zonas.head())

Columna isla GRAFCAN: nombre
Columna municipio GRAFCAN: nombre
Match isla %: 100.0
Match municipio %: 100.0


,nombre_zona,municipio,isla,lat,lon,nombre_zona_norm,municipio_norm,isla_norm,tipo_zona,geometry,distance_to_isla_m,spatial_match_isla,isla_spatial,distance_to_municipio_m,spatial_match_municipio,municipio_spatial
0,El Puertito,Güímar,Tenerife,28.2923,-16.3766,EL PUERTITO,GUIMAR,TENERIFE,playa,POINT (-16.3766 28.2923),0.0,True,TENERIFE,0.0,True,GÜÍMAR
1,La Restinga,El Pinar de El Hierro,El Hierro,27.6408,-17.9799,LA RESTINGA,EL PINAR DE EL HIERRO,EL HIERRO,playa,POINT (-17.9799 27.6408),0.0,True,EL HIERRO,0.0,True,EL PINAR
2,Arenas Blancas,Frontera,El Hierro,27.7667,-18.1218,ARENAS BLANCAS,FRONTERA,EL HIERRO,playa,POINT (-18.1218 27.7667),0.0,True,EL HIERRO,0.0,True,FRONTERA
3,El Verodal,Frontera,El Hierro,27.7471,-18.1512,EL VERODAL,FRONTERA,EL HIERRO,playa,POINT (-18.1512 27.7471),0.0,True,EL HIERRO,0.0,True,FRONTERA
4,Charco Azul,Frontera,El Hierro,27.7563,-18.0990,CHARCO AZUL,FRONTERA,EL HIERRO,playa,POINT (-18.099 27.7563),0.0,True,EL HIERRO,0.0,True,FRONTERA


# Crear zona_id

In [13]:
ISLAND_ALIASES = {
    "GRAN CANARIA": "GC",
    "TENERIFE": "TF",
    "LANZAROTE": "LZ",
    "FUERTEVENTURA": "FV",
    "LA PALMA": "LP",
    "LA GOMERA": "LG",
    "EL HIERRO": "EH",
    "GRACIOSA": "GR",
    "LA GRACIOSA": "GR",
    "ALEGRANZA": "LZ",
}


def island_abbr(isla):
    isla_norm = normalize_text(isla)

    if pd.isna(isla_norm):
        return "UNK"

    return ISLAND_ALIASES.get(isla_norm, slugify(isla_norm)[:3])


gdf_zonas["nombre_zona_norm"] = gdf_zonas["nombre_zona"].map(normalize_text)
gdf_zonas["isla_norm"] = gdf_zonas["isla"].map(normalize_text)

gdf_zonas["zona_id"] = gdf_zonas.apply(
    lambda r: f"CAN_{island_abbr(r['isla_norm'])}_{slugify(r['nombre_zona_norm'])}",
    axis=1,
)

duplicated = gdf_zonas["zona_id"].duplicated(keep=False)

if duplicated.any():
    gdf_zonas.loc[duplicated, "zona_id"] = (
        gdf_zonas.loc[duplicated, "zona_id"]
        + "_"
        + gdf_zonas.loc[duplicated].groupby("zona_id").cumcount().astype(str)
    )

print("Zonas únicas:", gdf_zonas["zona_id"].nunique())
print("Filas:", len(gdf_zonas))

display(gdf_zonas[["zona_id", "nombre_zona", "isla", "municipio", "lat", "lon"]].head())

Zonas únicas: 561
Filas: 561


,zona_id,nombre_zona,isla,municipio,lat,lon
0,CAN_TF_EL_PUERTITO_0,El Puertito,Tenerife,Güímar,28.2923,-16.3766
1,CAN_EH_LA_RESTINGA,La Restinga,El Hierro,El Pinar de El Hierro,27.6408,-17.9799
2,CAN_EH_ARENAS_BLANCAS,Arenas Blancas,El Hierro,Frontera,27.7667,-18.1218
3,CAN_EH_EL_VERODAL,El Verodal,El Hierro,Frontera,27.7471,-18.1512
4,CAN_EH_CHARCO_AZUL_0,Charco Azul,El Hierro,Frontera,27.7563,-18.0990


# Orientación y exposición costera inicial

In [14]:
def estimate_exposure(row):
    lon = row["lon"]
    lat = row["lat"]

    exposicion_norte = int(lat >= 28.2)
    exposicion_oeste = int(lon <= -16.2)
    exposicion_este = int(lon > -16.2)

    exposicion_swell_nw = int(exposicion_norte or exposicion_oeste)
    exposicion_swell_ne = int(exposicion_norte or exposicion_este)

    if exposicion_norte and exposicion_oeste:
        orientacion = "NW"
    elif exposicion_norte and exposicion_este:
        orientacion = "NE"
    elif exposicion_oeste:
        orientacion = "W"
    elif exposicion_este:
        orientacion = "E"
    else:
        orientacion = "S"

    return pd.Series(
        {
            "orientacion_costa": orientacion,
            "exposicion_norte": exposicion_norte,
            "exposicion_oeste": exposicion_oeste,
            "exposicion_este": exposicion_este,
            "exposicion_swell_nw": exposicion_swell_nw,
            "exposicion_swell_ne": exposicion_swell_ne,
        }
    )


exposure_cols = gdf_zonas.apply(estimate_exposure, axis=1)

gdf_zonas = pd.concat(
    [
        gdf_zonas.drop(
            columns=[
                "orientacion_costa",
                "exposicion_norte",
                "exposicion_oeste",
                "exposicion_este",
                "exposicion_swell_nw",
                "exposicion_swell_ne",
            ],
            errors="ignore",
        ),
        exposure_cols,
    ],
    axis=1,
)

display(
    gdf_zonas[
        [
            "zona_id",
            "nombre_zona",
            "isla",
            "municipio",
            "lat",
            "lon",
            "orientacion_costa",
            "exposicion_norte",
            "exposicion_oeste",
            "exposicion_este",
            "exposicion_swell_nw",
            "exposicion_swell_ne",
        ]
    ].head()
)

,zona_id,nombre_zona,isla,municipio,lat,lon,orientacion_costa,exposicion_norte,exposicion_oeste,exposicion_este,exposicion_swell_nw,exposicion_swell_ne
0,CAN_TF_EL_PUERTITO_0,El Puertito,Tenerife,Güímar,28.2923,-16.3766,NW,1,1,0,1,1
1,CAN_EH_LA_RESTINGA,La Restinga,El Hierro,El Pinar de El Hierro,27.6408,-17.9799,W,0,1,0,1,0
2,CAN_EH_ARENAS_BLANCAS,Arenas Blancas,El Hierro,Frontera,27.7667,-18.1218,W,0,1,0,1,0
3,CAN_EH_EL_VERODAL,El Verodal,El Hierro,Frontera,27.7471,-18.1512,W,0,1,0,1,0
4,CAN_EH_CHARCO_AZUL_0,Charco Azul,El Hierro,Frontera,27.7563,-18.0990,W,0,1,0,1,0


# Vulnerabilidad costera MITECO

In [15]:
gdf_zonas["vulnerabilidad_costera"] = "no_disponible"

if PATH_MITECO is not None:
    gdf_zonas["vulnerabilidad_source"] = str(PATH_MITECO)
else:
    gdf_zonas["vulnerabilidad_source"] = "MITECO_reference_pending_structured_layer"

display(
    gdf_zonas[
        [
            "zona_id",
            "nombre_zona",
            "vulnerabilidad_costera",
            "vulnerabilidad_source",
        ]
    ].head()
)

,zona_id,nombre_zona,vulnerabilidad_costera,vulnerabilidad_source
0,CAN_TF_EL_PUERTITO_0,El Puertito,no_disponible,/content/drive/MyDrive/AI Projects/DeepWave Ca...
1,CAN_EH_LA_RESTINGA,La Restinga,no_disponible,/content/drive/MyDrive/AI Projects/DeepWave Ca...
2,CAN_EH_ARENAS_BLANCAS,Arenas Blancas,no_disponible,/content/drive/MyDrive/AI Projects/DeepWave Ca...
3,CAN_EH_EL_VERODAL,El Verodal,no_disponible,/content/drive/MyDrive/AI Projects/DeepWave Ca...
4,CAN_EH_CHARCO_AZUL_0,Charco Azul,no_disponible,/content/drive/MyDrive/AI Projects/DeepWave Ca...


# Construir tabla final beach_geography

In [16]:
final_cols = [
    "zona_id",
    "nombre_zona",
    "isla",
    "municipio",
    "lat",
    "lon",
    "tipo_zona",
    "orientacion_costa",
    "exposicion_norte",
    "exposicion_oeste",
    "exposicion_este",
    "exposicion_swell_nw",
    "exposicion_swell_ne",
    "vulnerabilidad_costera",
    "vulnerabilidad_source",
    "spatial_match_isla",
    "spatial_match_municipio",
]

beach_geography = pd.DataFrame(
    gdf_zonas[final_cols].copy()
)

for col in [
    "exposicion_norte",
    "exposicion_oeste",
    "exposicion_este",
    "exposicion_swell_nw",
    "exposicion_swell_ne",
]:
    beach_geography[col] = beach_geography[col].fillna(0).astype("int8")

beach_geography["spatial_match_isla"] = beach_geography["spatial_match_isla"].fillna(False).astype(bool)
beach_geography["spatial_match_municipio"] = beach_geography["spatial_match_municipio"].fillna(False).astype(bool)

beach_geography = beach_geography.drop_duplicates(subset=["zona_id"]).reset_index(drop=True)

print("Shape final beach_geography:", beach_geography.shape)

display(beach_geography.head())

Shape final beach_geography: (561, 17)


,zona_id,nombre_zona,isla,municipio,lat,lon,tipo_zona,orientacion_costa,exposicion_norte,exposicion_oeste,exposicion_este,exposicion_swell_nw,exposicion_swell_ne,vulnerabilidad_costera,vulnerabilidad_source,spatial_match_isla,spatial_match_municipio
0,CAN_TF_EL_PUERTITO_0,El Puertito,Tenerife,Güímar,28.2923,-16.3766,playa,NW,1,1,0,1,1,no_disponible,/content/drive/MyDrive/AI Projects/DeepWave Ca...,True,True
1,CAN_EH_LA_RESTINGA,La Restinga,El Hierro,El Pinar de El Hierro,27.6408,-17.9799,playa,W,0,1,0,1,0,no_disponible,/content/drive/MyDrive/AI Projects/DeepWave Ca...,True,True
2,CAN_EH_ARENAS_BLANCAS,Arenas Blancas,El Hierro,Frontera,27.7667,-18.1218,playa,W,0,1,0,1,0,no_disponible,/content/drive/MyDrive/AI Projects/DeepWave Ca...,True,True
3,CAN_EH_EL_VERODAL,El Verodal,El Hierro,Frontera,27.7471,-18.1512,playa,W,0,1,0,1,0,no_disponible,/content/drive/MyDrive/AI Projects/DeepWave Ca...,True,True
4,CAN_EH_CHARCO_AZUL_0,Charco Azul,El Hierro,Frontera,27.7563,-18.0990,playa,W,0,1,0,1,0,no_disponible,/content/drive/MyDrive/AI Projects/DeepWave Ca...,True,True


# Reporte de calidad

In [17]:
quality_summary = pd.DataFrame(
    {
        "metric": [
            "rows",
            "unique_zona_id",
            "missing_nombre_zona",
            "missing_isla",
            "missing_municipio",
            "missing_lat",
            "missing_lon",
            "duplicated_zona_id",
            "outside_bbox",
            "spatial_match_isla_pct",
            "spatial_match_municipio_pct",
        ],
        "value": [
            len(beach_geography),
            beach_geography["zona_id"].nunique(),
            int(beach_geography["nombre_zona"].isna().sum()),
            int(beach_geography["isla"].isna().sum()),
            int(beach_geography["municipio"].isna().sum()),
            int(beach_geography["lat"].isna().sum()),
            int(beach_geography["lon"].isna().sum()),
            int(beach_geography["zona_id"].duplicated().sum()),
            int(
                (
                    ~beach_geography["lat"].between(BBOX_CANARIAS["lat_min"], BBOX_CANARIAS["lat_max"])
                    | ~beach_geography["lon"].between(BBOX_CANARIAS["lon_min"], BBOX_CANARIAS["lon_max"])
                ).sum()
            ),
            float(beach_geography["spatial_match_isla"].mean() * 100),
            float(beach_geography["spatial_match_municipio"].mean() * 100),
        ],
    }
)

missing_by_column = (
    beach_geography.isna()
    .mean()
    .mul(100)
    .reset_index()
    .rename(columns={"index": "column", 0: "missing_pct"})
)

print("Resumen de calidad:")
display(quality_summary)

print("Nulos por columna:")
display(missing_by_column)

Resumen de calidad:


,metric,value
0,rows,561.0
1,unique_zona_id,561.0
2,missing_nombre_zona,0.0
3,missing_isla,0.0
4,missing_municipio,0.0
5,missing_lat,0.0
6,missing_lon,0.0
7,duplicated_zona_id,0.0
8,outside_bbox,0.0
9,spatial_match_isla_pct,100.0


Nulos por columna:


,column,missing_pct
0,zona_id,0.0
1,nombre_zona,0.0
2,isla,0.0
3,municipio,0.0
4,lat,0.0
5,lon,0.0
6,tipo_zona,0.0
7,orientacion_costa,0.0
8,exposicion_norte,0.0
9,exposicion_oeste,0.0


In [18]:
outside_bbox_value = int(
    quality_summary.loc[
        quality_summary["metric"] == "outside_bbox",
        "value"
    ].iloc[0]
)

duplicated_zona_id_value = int(
    quality_summary.loc[
        quality_summary["metric"] == "duplicated_zona_id",
        "value"
    ].iloc[0]
)

missing_lat_value = int(
    quality_summary.loc[
        quality_summary["metric"] == "missing_lat",
        "value"
    ].iloc[0]
)

missing_lon_value = int(
    quality_summary.loc[
        quality_summary["metric"] == "missing_lon",
        "value"
    ].iloc[0]
)

spatial_match_isla_pct = float(
    quality_summary.loc[
        quality_summary["metric"] == "spatial_match_isla_pct",
        "value"
    ].iloc[0]
)

spatial_match_municipio_pct = float(
    quality_summary.loc[
        quality_summary["metric"] == "spatial_match_municipio_pct",
        "value"
    ].iloc[0]
)

unknown_municipios = int(
    (beach_geography["municipio"] == "MUNICIPIO_DESCONOCIDO").sum()
)

print("outside_bbox:", outside_bbox_value)
print("duplicated_zona_id:", duplicated_zona_id_value)
print("missing_lat:", missing_lat_value)
print("missing_lon:", missing_lon_value)
print("spatial_match_isla_pct:", spatial_match_isla_pct)
print("spatial_match_municipio_pct:", spatial_match_municipio_pct)
print("municipios desconocidos:", unknown_municipios)

if outside_bbox_value > 0:
    raise ValueError(
        f"Hay {outside_bbox_value} zonas fuera del bbox de Canarias. No guardar todavía."
    )

if duplicated_zona_id_value > 0:
    raise ValueError(
        f"Hay {duplicated_zona_id_value} zona_id duplicados. No guardar todavía."
    )

if missing_lat_value > 0 or missing_lon_value > 0:
    raise ValueError(
        "Hay coordenadas lat/lon nulas. No guardar todavía."
    )

if unknown_municipios > 0:
    raise ValueError(
        f"Hay {unknown_municipios} municipios desconocidos. No guardar todavía."
    )

if spatial_match_isla_pct == 0:
    raise ValueError(
        "El spatial join con islas ha dado 0%. No guardar todavía."
    )

print("Validación final correcta. Se puede guardar beach_geography.parquet.")

outside_bbox: 0
duplicated_zona_id: 0
missing_lat: 0
missing_lon: 0
spatial_match_isla_pct: 100.0
spatial_match_municipio_pct: 100.0
municipios desconocidos: 0
Validación final correcta. Se puede guardar beach_geography.parquet.


# Guardar Parquet y reportes

In [19]:
OUT_PARQUET = OUT_DIR / "beach_geography.parquet"
OUT_QC_SUMMARY = QC_DIR / "quality_beach_geography_summary.csv"
OUT_QC_MISSING = QC_DIR / "quality_beach_geography_missing_by_column.csv"

beach_geography.to_parquet(
    OUT_PARQUET,
    index=False,
    engine="pyarrow",
    compression="snappy",
)

quality_summary.to_csv(OUT_QC_SUMMARY, index=False)
missing_by_column.to_csv(OUT_QC_MISSING, index=False)

print("Archivos guardados correctamente:")
print(OUT_PARQUET)
print(OUT_QC_SUMMARY)
print(OUT_QC_MISSING)

Archivos guardados correctamente:
/content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/beach_geography/beach_geography.parquet
/content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_quality_reports/quality_beach_geography_summary.csv
/content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_quality_reports/quality_beach_geography_missing_by_column.csv


# Comprobación final

In [20]:
test = pd.read_parquet(OUT_PARQUET)

print("Parquet leído correctamente.")
print("Shape:", test.shape)
print("Columnas:")
print(test.columns.tolist())

display(test.head())

Parquet leído correctamente.
Shape: (561, 17)
Columnas:
['zona_id', 'nombre_zona', 'isla', 'municipio', 'lat', 'lon', 'tipo_zona', 'orientacion_costa', 'exposicion_norte', 'exposicion_oeste', 'exposicion_este', 'exposicion_swell_nw', 'exposicion_swell_ne', 'vulnerabilidad_costera', 'vulnerabilidad_source', 'spatial_match_isla', 'spatial_match_municipio']


,zona_id,nombre_zona,isla,municipio,lat,lon,tipo_zona,orientacion_costa,exposicion_norte,exposicion_oeste,exposicion_este,exposicion_swell_nw,exposicion_swell_ne,vulnerabilidad_costera,vulnerabilidad_source,spatial_match_isla,spatial_match_municipio
0,CAN_TF_EL_PUERTITO_0,El Puertito,Tenerife,Güímar,28.2923,-16.3766,playa,NW,1,1,0,1,1,no_disponible,/content/drive/MyDrive/AI Projects/DeepWave Ca...,True,True
1,CAN_EH_LA_RESTINGA,La Restinga,El Hierro,El Pinar de El Hierro,27.6408,-17.9799,playa,W,0,1,0,1,0,no_disponible,/content/drive/MyDrive/AI Projects/DeepWave Ca...,True,True
2,CAN_EH_ARENAS_BLANCAS,Arenas Blancas,El Hierro,Frontera,27.7667,-18.1218,playa,W,0,1,0,1,0,no_disponible,/content/drive/MyDrive/AI Projects/DeepWave Ca...,True,True
3,CAN_EH_EL_VERODAL,El Verodal,El Hierro,Frontera,27.7471,-18.1512,playa,W,0,1,0,1,0,no_disponible,/content/drive/MyDrive/AI Projects/DeepWave Ca...,True,True
4,CAN_EH_CHARCO_AZUL_0,Charco Azul,El Hierro,Frontera,27.7563,-18.0990,playa,W,0,1,0,1,0,no_disponible,/content/drive/MyDrive/AI Projects/DeepWave Ca...,True,True
